In [1]:
import json, shutil, os, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from threadpoolctl import threadpool_limits
threadpool_limits(limits=1)
from itertools import product

In [2]:
TASK = Path.cwd().parent
INPUT, OUTPUT = TASK / 'input', TASK / 'output'

In [3]:
config = json.loads((INPUT / 'config.json').read_text())
SEED = config['seed']
rng = np.random.default_rng(SEED)

In [4]:
for cell in json.loads((INPUT / 'helpers.ipynb').read_text())['cells']:
    exec(''.join(cell['source']), globals())
carry(INPUT, OUTPUT)

In [5]:
queries = pd.read_csv(INPUT / 'queries.csv')
blocks = np.load(INPUT / 'query_blocks.npz')['blocks']
truth = np.load(INPUT / 'truth.npz')['ancestry']
nets = load_nets(INPUT / 'trained_networks.npz')
marker_order = np.load(INPUT / 'marker_order.npy')

In [6]:
# independent correctness check against exhaustive path enumeration
toy = np.array([[0.1, 0.9, 0.8], [0.4, 0.2, 0.6], [0.8, 0.2, 0.1], [0.6, 0.5, 0.2]])
paths = np.array(list(product(range(3), repeat=len(toy))))
energy = toy[np.arange(len(toy))[None], paths].sum(axis=1) + 0.7 * np.sum(paths[:, 1:] != paths[:, :-1], axis=1)
chosen = viterbi(toy, 0.7)
assert np.array_equal(chosen, paths[energy.argmin()])
weights = softmax(-energy / 0.4)
brute_marginals = np.stack([[(weights * (paths[:, j] == k)).sum() for k in range(3)] for j in range(len(toy))])
assert np.allclose(posterior(toy, 0.7, 0.4), brute_marginals, atol=1e-12)
assert np.allclose(source_probabilities(posterior(np.repeat(toy, 3, axis=1), 0, 0.4)), posterior(toy, 0, 0.4), atol=1e-12)
assert boundary_counts(np.array([0, 0, 1, 1, 2]), np.array([0, 1, 1, 2, 2]), 1) == (2, 0, 0)
(OUTPUT / 'decoder_checks.json').write_text(json.dumps({'viterbi_matches_exhaustive': True,
    'posterior_max_error': float(np.abs(brute_marginals - posterior(toy, 0.7, 0.4)).max()),
    'one_to_one_boundary_matching': True}, indent=2))

128

In [7]:
panels = {}
for split in ['cal', 'test']:
    panels[f'{split}_exact'] = np.stack([block_vectors(nets[f'{split}_donor{k}'])[None] for k in range(3)])
    panels[f'{split}_proxy'] = np.stack([[block_vectors(nets[f'reference{j}{k}'])
        for j in range(config['references_per_source'])] for k in range(3)])
# normalization from proxy references only; no held-out query ancestry
centers = panels['cal_proxy'].mean(axis=1)
scale = np.mean([(centers[a] - centers[b]) ** 2 for a in range(3) for b in range(a)], axis=(0, 2))
scale = np.maximum(scale, 1e-12)
# publish the panel only after its compressed archive is complete
with (OUTPUT / 'reference_panels.tmp').open('wb') as handle:
    np.savez_compressed(handle, **panels, scale=scale)
    handle.flush()
    os.fsync(handle.fileno())
(OUTPUT / 'reference_panels.tmp').replace(OUTPUT / 'reference_panels.npz')

PosixPath('/workspace/scratch/25b3eb6f4388/03_neural_local_ancestry_PDP/d_fit/output/reference_panels.npz')

In [8]:
fit_rows, grid_rows = [], []
for panel in ['exact', 'proxy']:
    for markers in config['marker_counts']:
        indices = marker_order[:markers]
        for steps in config['finetune_steps']:
            selected = queries.query("split == 'cal' and steps == @steps and variant == 'aligned'")
            ids = selected.query_id.to_numpy()
            costs = np.stack([copy_cost(blocks[q], panels[f'cal_{panel}'], indices, scale) for q in ids])
            targets = truth[ids]
            best = None
            for lam, temperature in product(config['lambda_grid'], config['temperature_grid']):
                p = source_probabilities(posterior(costs, lam, temperature))
                nll = -np.log(np.maximum(p[np.arange(len(ids))[:, None], np.arange(config['blocks'])[None], targets], 1e-300)).mean()
                row = {'panel': panel, 'markers': markers, 'steps': steps, 'lambda': lam,
                       'temperature': temperature, 'calibration_nll': float(nll)}
                grid_rows.append(row)
                if best is None or nll < best['calibration_nll'] - 1e-12:
                    best = row.copy()
            best['rejection_threshold'] = float(max(np.quantile(costs.min(axis=2), config['rejection_quantile']), 1e-10))
            best['calibration_accuracy'] = float(np.mean(viterbi(costs, best['lambda']) // (costs.shape[-1] // 3) == targets))
            best['calibration_mosaics'] = len(ids)
            fit_rows.append(best)
pd.DataFrame(fit_rows).to_csv(OUTPUT / 'fitted_parameters.csv', index=False)
pd.DataFrame(grid_rows).to_csv(OUTPUT / 'calibration_grid.csv', index=False)
pd.DataFrame(fit_rows).groupby(['panel', 'markers'])[['lambda', 'temperature', 'calibration_accuracy']].mean()

lambda  temperature  calibration_accuracy
panel markers                                           
exact 1        0.5625      1.56875              0.631944
      4        0.1125      1.14375              0.786458
      16       0.0625      1.07500              0.835648
      64       0.0125      1.03750              0.887731
      600      0.0500      0.52500              0.897569
proxy 1        1.0000      1.62500              0.489005
      4        0.4250      1.31250              0.540509
      16       0.1125      1.10000              0.740162
      64       0.0750      1.03750              0.814236
      600      0.0250      0.52500              0.821759